# Auto search — the system picks the keywords (first step of the daily job)

Design: Claude Doc "Project 1 — Tự động search keyword" (datle approved 2026-10-31).

0. **Followed channels** (`search_task.run_channel_uploads`, source A, added 2026-10-31): newest uploads of known food channels, 1 unit per channel (see below).
1. **Plan** (`keywords.plan`): seed the grid dish × district of HCMC into `public.search_keywords` (once), add `"<restaurant> review"` keywords for well-rated restaurants that only one channel reviewed (from `gold.agg_restaurant_ranking`), recompute every keyword's priority (thin (district, cuisine) on the dashboard first, never-searched first, keywords that found new videos last time first).
2. **Search** (`search_task.run_auto`): the `auto_search_count` due YouTube keywords with the highest priority, `max_results` each, `source = search_auto`. First run of a keyword: `order = relevance`; later runs: `order = date` + `publishedAfter` = previous search, so only new uploads cost quota. Each keyword then rests 7 days (doubled up to 60 while it finds nothing new).
3. **Google Maps** (`search_task.run_auto_places`, added 2026-10-31): the `auto_places_count` (40) due keywords of the same grid in mode `places`, 1 **Text Search Pro** page (20 places, 1 call) each, `source = search_auto`; cooldown 30 days (doubled up to 90). Since 2026-11-01 these calls are Pro (no rating / price / hours / phone / website, capped at 1,500 Pro calls / month); `stg_restaurant_place_details` fills those fields later. Manual searches stay Enterprise (800 / month). Places-only restaurants reach `dim_restaurant` through `silver.stg_places`; well-rated ones (≥ 4.3, ≥ 200 reviews) become YouTube verify keywords.

**Free quota:** up to 100 channels × 1 unit + new videos' `videos.list` (1 unit per 50), then 20 searches × 100 units = 2,000 of the 10,000 free YouTube units per day; a quota error stops the loop without failing. Google Maps: at most 450 auto calls / month, inside the shared cap of 800 Text Search Enterprise calls (1,000 free). 

The task never fails the job: an error is printed and the daily run goes on with refresh.

In [ ]:
%pip install --quiet "psycopg>=3.2.0" "httpx>=0.28.0" "python-dotenv>=1.0.0"


In [ ]:
# --- Config ---
import os
import sys

for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
GOLD = dbutils.widgets.get("gold")
dbutils.widgets.text("auto_search_count", "20")
dbutils.widgets.text("auto_search_max_results", "25")
dbutils.widgets.text("auto_channel_limit", "100")
dbutils.widgets.text("auto_places_count", "40")
dbutils.widgets.dropdown("dry_run", "false", ["false", "true"])
dbutils.widgets.text("databricks_run_id", "")

COUNT = int(dbutils.widgets.get("auto_search_count"))
MAX_RESULTS = int(dbutils.widgets.get("auto_search_max_results"))
CHANNEL_LIMIT = int(dbutils.widgets.get("auto_channel_limit"))
PLACES_COUNT = int(dbutils.widgets.get("auto_places_count"))
DRY_RUN = dbutils.widgets.get("dry_run") == "true"
RUN_ID = dbutils.widgets.get("databricks_run_id")
RUN_ID = int(RUN_ID) if RUN_ID.isdigit() else None

os.environ.update({
    "PGHOST": "ep-still-butterfly-d8fd3r70.database.us-east-2.cloud.databricks.com",
    "PGDATABASE": "databricks_postgres",
    "PGUSER": "admin",
    "PGPORT": "5432",
    "PGSSLMODE": "require",
    "APP_SCHEMA": "public",
    "PGPASSWORD": dbutils.secrets.get("youtube", "pg_password"),
    "YOUTUBE_API_KEY": dbutils.secrets.get("youtube", "api_key"),
    # FIPS: use psycopg's pure-Python implementation, not the bundled OpenSSL of psycopg[binary]
    "PSYCOPG_IMPL": "python",
    "MAPS_API_KEY": dbutils.secrets.get("maps", "api_key"),
})

notebook_dir = os.path.dirname(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get())
app_dir = os.path.normpath(f"/Workspace{notebook_dir}/../../app")  # src/notebooks/refresh -> src/app
sys.path.insert(0, app_dir)
print(f"App code: {app_dir} · gold = {GOLD} · {CHANNEL_LIMIT} channel(s) · {COUNT} keyword(s) × {MAX_RESULTS} results · {PLACES_COUNT} Google Maps keyword(s) · dry_run = {DRY_RUN}")

## Source A: new uploads of followed food channels

A channel is followed once `gold.dim_video` has at least one of its videos classified as a food review (`video_type` set by `stg_video_classification` and `is_food_review`). For each channel (never checked first, at most `auto_channel_limit`), one `playlistItems.list` call on its uploads playlist (**1 unit**, vs 100 for `search.list`) lists the 50 newest uploads; uploads since the previous check (first check: last 90 days) that are not in Postgres yet are saved with `source = search_channel`. Non-food uploads are filtered later by `stg_video_classification`.

In [ ]:
# --- Source A: followed food channels (never fails the job) ---
import json
import search_task

DIM_VIDEO, DIM_CHANNEL = f"{GOLD}.dim_video", f"{GOLD}.dim_channel"
channels = []
if spark.catalog.tableExists(DIM_VIDEO) and "video_type" in spark.table(DIM_VIDEO).columns:
    title_sql = (f"LEFT JOIN {DIM_CHANNEL} c ON c.channel_id = v.channel_id" if spark.catalog.tableExists(DIM_CHANNEL)
                 else "LEFT JOIN (SELECT NULL AS channel_id, NULL AS channel_title) c ON false")
    channels = [r.asDict() for r in spark.sql(f"""
        SELECT v.channel_id, MAX(c.channel_title) AS channel_title, COUNT(*) AS food_review_videos
        FROM {DIM_VIDEO} v {title_sql}
        WHERE v.video_key <> -1 AND v.channel_id IS NOT NULL
          AND v.video_type IS NOT NULL AND v.is_food_review   -- classified, not the default
        GROUP BY v.channel_id
    """).collect()]
print(f"{len(channels)} food channel(s) in {DIM_VIDEO}")

try:
    channel_result = search_task.run_channel_uploads(channels, limit=CHANNEL_LIMIT, dry_run=DRY_RUN,
                                                     databricks_run_id=RUN_ID)
except Exception as error:  # noqa: BLE001 - the keyword search and refresh must still run
    channel_result = {"error": str(error)[:2000]}
    print(f"⚠️ channel uploads failed, auto_search goes on: {error}")

In [ ]:
# --- Restaurants on the dashboard (input of the planner); empty before the first Gold run ---
AGG = f"{GOLD}.agg_restaurant_ranking"
COLUMNS = ["restaurant_name", "district", "city", "cuisine", "legit_label", "channel_count",
           "google_rating", "google_rating_count", "has_youtube_review"]
restaurants = []
if spark.catalog.tableExists(AGG):
    available = set(spark.table(AGG).columns)
    select = ", ".join(c if c in available else f"NULL AS {c}" for c in COLUMNS)
    restaurants = [r.asDict() for r in spark.sql(
        f"SELECT {select} FROM {AGG} WHERE city IS NULL OR city LIKE '%Hồ Chí Minh%'").collect()]
print(f"{len(restaurants)} HCMC restaurant(s) read from {AGG}")

In [ ]:
# --- Plan + search (never fails the job) ---
try:
    result = search_task.run_auto(limit=COUNT, max_results=MAX_RESULTS, restaurants=restaurants,
                                  dry_run=DRY_RUN, databricks_run_id=RUN_ID, places_limit=PLACES_COUNT)
except Exception as error:  # noqa: BLE001 - the daily run must go on with refresh
    result = {"error": str(error)[:2000]}
    print(f"⚠️ auto_search failed, the daily job goes on: {error}")
for kw in result.get("keywords", []):
    print(f"  {kw['origin']:<7} {kw['keyword']}: {kw['videos']} video(s), {kw['new_videos']} new")
for kw in (result.get("places") or {}).get("keywords", []):
    print(f"  places  {kw['keyword']}: {kw['places']} place(s), {kw['new_places']} new")
result["channel_uploads"] = channel_result
dbutils.notebook.exit(json.dumps(result, ensure_ascii=False, default=str))